# Gaussian Mixture Models — Full Project Notebook

In [ ]:
import numpy as np
import scipy
import scipy.special
import scipy.optimize
import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
# ── Basic utility functions ───────────────────────────────────────────────────
def vcol(x):  return x.reshape(x.size, 1)
def vrow(x):  return x.reshape(1, x.size)

def split_db_2to1(D, L, seed=0):
    nTrain = int(D.shape[1] * 2.0 / 3.0)
    np.random.seed(seed)
    idx   = np.random.permutation(D.shape[1])
    idxTrain = idx[:nTrain];  idxTest = idx[nTrain:]
    DTR = D[:, idxTrain];     DVAL = D[:, idxTest]
    LTR = L[idxTrain];        LVAL = L[idxTest]
    return DTR, LTR, DVAL, LVAL

def computemuC(D):
    mu = vcol(D.mean(1))
    C  = (D - mu) @ (D - mu).T / float(D.shape[1])
    return mu, C


## 1. GMM Density

In [ ]:
def logpdf_GAU_ND(X, mu, C):
    P = np.linalg.inv(C)
    return (-0.5 * X.shape[0] * np.log(2.0 * np.pi)
            - 0.5 * np.linalg.slogdet(C)[1]
            - 0.5 * ((X - mu) * (P @ (X - mu))).sum(0))

def logpdf_GMM(X, gmm):
    S = []
    for w, mu, C in gmm:
        S.append(logpdf_GAU_ND(X, mu, C) + np.log(w))
    S = np.vstack(S)
    return scipy.special.logsumexp(S, axis=0)

def smooth_covariance_matrix(C, psi):
    U, s, _ = np.linalg.svd(C)
    s[s < psi] = psi
    return U @ (vcol(s) * U.T)


## 2. EM Algorithm for GMM

In [ ]:
def train_GMM_EM_Iteration(X, gmm, covType='full', psiEig=None):
    assert covType.lower() in ['full', 'diagonal', 'tied']
    S = []
    for w, mu, C in gmm:
        S.append(logpdf_GAU_ND(X, mu, C) + np.log(w))
    S        = np.vstack(S)
    logdens  = scipy.special.logsumexp(S, axis=0)
    gammaAll = np.exp(S - logdens)

    gmmNew = []
    for gIdx in range(len(gmm)):
        gamma = gammaAll[gIdx]
        Z     = gamma.sum()
        F     = vcol((vrow(gamma) * X).sum(1))
        Sg    = (vrow(gamma) * X) @ X.T
        wNew  = Z / X.shape[1]
        muNew = F / Z
        CNew  = Sg / Z - muNew @ muNew.T
        if covType.lower() == 'diagonal':
            CNew = CNew * np.eye(X.shape[0])
        if psiEig is not None:
            CNew = smooth_covariance_matrix(CNew, psiEig)
        gmmNew.append((wNew, muNew, CNew))

    if covType.lower() == 'tied':
        CTied = sum(w * C for w, _, C in gmmNew)
        gmmNew = [(w, mu,
                   smooth_covariance_matrix(CTied, psiEig) if psiEig is not None else CTied)
                  for w, mu, _ in gmmNew]

    return gmmNew

def train_GMM_EM(X, gmm, covType='full', psiEig=None, epsLLAverage=1e-6, verbose=False):
    llOld   = logpdf_GMM(X, gmm).mean()
    llDelta = None
    it = 0
    while llDelta is None or llDelta > epsLLAverage:
        gmmUpd  = train_GMM_EM_Iteration(X, gmm, covType=covType, psiEig=psiEig)
        llUpd   = logpdf_GMM(X, gmmUpd).mean()
        llDelta = llUpd - llOld
        gmm     = gmmUpd
        llOld   = llUpd
        it += 1
        if verbose:
            print(f'EM iter {it:3d} – avg ll {llUpd:.8e}')
    return gmm


## 3. LBG Algorithm

In [ ]:
def split_GMM_LBG(gmm, alpha=0.1):
    gmmOut = []
    for w, mu, C in gmm:
        U, s, _ = np.linalg.svd(C)
        d = U[:, 0:1] * s[0]**0.5 * alpha
        gmmOut.append((0.5 * w, mu - d, C))
        gmmOut.append((0.5 * w, mu + d, C))
    return gmmOut

def train_GMM_LBG_EM(X, numComponents, covType='full', psiEig=None,
                     epsLLAverage=1e-6, lbgAlpha=0.1, verbose=False):
    mu, C = computemuC(X)
    if covType.lower() == 'diagonal':
        C = C * np.eye(X.shape[0])
    if psiEig is not None:
        C = smooth_covariance_matrix(C, psiEig)
    gmm = [(1.0, mu, C)]
    while len(gmm) < numComponents:
        gmm = split_GMM_LBG(gmm, alpha=lbgAlpha)
        gmm = train_GMM_EM(X, gmm, covType=covType, psiEig=psiEig,
                           epsLLAverage=epsLLAverage, verbose=verbose)
    return gmm


## 4. Bayes Risk Functions

In [ ]:
def compute_confusion_matrix(predictedLabels, classLabels):
    nClasses = classLabels.max() + 1
    M = np.zeros((nClasses, nClasses), dtype=np.int32)
    for i in range(classLabels.size):
        M[predictedLabels[i], classLabels[i]] += 1
    return M

def compute_optimal_Bayes_binary(llr, prior, Cfn, Cfp):
    th = -np.log(prior * Cfn / ((1 - prior) * Cfp))
    return np.int32(llr > th)

def compute_empirical_Bayes_risk_binary(predictedLabels, classLabels, prior, Cfn, Cfp, normalize=True):
    M   = compute_confusion_matrix(predictedLabels, classLabels)
    Pfn = M[0, 1] / (M[0, 1] + M[1, 1])
    Pfp = M[1, 0] / (M[0, 0] + M[1, 0])
    bayesError = prior * Cfn * Pfn + (1 - prior) * Cfp * Pfp
    if normalize:
        return bayesError / np.minimum(prior * Cfn, (1 - prior) * Cfp)
    return bayesError

def compute_actDCF_binary_fast(llr, classLabels, prior, Cfn, Cfp):
    predictedLabels = compute_optimal_Bayes_binary(llr, prior, Cfn, Cfp)
    return compute_empirical_Bayes_risk_binary(predictedLabels, classLabels, prior, Cfn, Cfp, normalize=True)

def compute_PfnPfp_allThresholds_fast(llr, classLabels):
    llrSorter       = np.argsort(llr)
    llrSorted       = llr[llrSorter]
    classLabelsSorted = classLabels[llrSorter]
    Pfp = []; Pfn = []
    nTrue  = (classLabelsSorted == 1).sum()
    nFalse = (classLabelsSorted == 0).sum()
    nFalseNegative = 0;  nFalsePositive = nFalse
    Pfn.append(nFalseNegative / nTrue);  Pfp.append(nFalsePositive / nFalse)
    for idx in range(len(llrSorted)):
        if classLabelsSorted[idx] == 1: nFalseNegative += 1
        if classLabelsSorted[idx] == 0: nFalsePositive -= 1
        Pfn.append(nFalseNegative / nTrue);  Pfp.append(nFalsePositive / nFalse)
    llrSorted  = np.concatenate([-np.array([np.inf]), llrSorted])
    PfnOut = []; PfpOut = []; thresholdsOut = []
    for idx in range(len(llrSorted)):
        if idx == len(llrSorted) - 1 or llrSorted[idx + 1] != llrSorted[idx]:
            PfnOut.append(Pfn[idx]);  PfpOut.append(Pfp[idx])
            thresholdsOut.append(llrSorted[idx])
    return np.array(PfnOut), np.array(PfpOut), np.array(thresholdsOut)

def compute_minDCF_binary_fast(llr, classLabels, prior, Cfn, Cfp, returnThreshold=False):
    Pfn, Pfp, th = compute_PfnPfp_allThresholds_fast(llr, classLabels)
    dcf = (prior * Cfn * Pfn + (1 - prior) * Cfp * Pfp) / np.minimum(prior * Cfn, (1 - prior) * Cfp)
    idx = np.argmin(dcf)
    if returnThreshold:
        return dcf[idx], th[idx]
    return dcf[idx]

def bayes_error_plot_data(llr, classLabels, Cfn=1.0, Cfp=1.0, effPriorLogOdds=None):
    if effPriorLogOdds is None:
        effPriorLogOdds = np.linspace(-4, 4, 100)
    minDCF_vals = []; actDCF_vals = []
    for lo in effPriorLogOdds:
        prior = 1.0 / (1.0 + np.exp(-lo))
        minDCF_vals.append(compute_minDCF_binary_fast(llr, classLabels, prior, Cfn, Cfp))
        actDCF_vals.append(compute_actDCF_binary_fast(llr, classLabels, prior, Cfn, Cfp))
    return np.array(effPriorLogOdds), np.array(minDCF_vals), np.array(actDCF_vals)


## 5. Load Project Data

In [ ]:
def load_data(path):
    rows = []
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            values = [float(x.strip()) for x in line.split(',')]
            rows.append(values)
    A = np.array(rows, dtype=float)
    X = A[:, :-1].T
    L = A[:,  -1].astype(int)
    return X, L

# ── adjust the path if needed ──
X, L = load_data('../../../Project/trainData.txt')
print('Dataset shape:', X.shape)
print('Class 0 samples:', (L == 0).sum())
print('Class 1 samples:', (L == 1).sum())

DTR, LTR, DVAL, LVAL = split_db_2to1(X, L, seed=0)
print('Training set:', DTR.shape)
print('Validation set:', DVAL.shape)


## 6. Initial Inspection

In [ ]:
mu0, C0 = computemuC(DTR[:, LTR == 0])
mu1, C1 = computemuC(DTR[:, LTR == 1])
print('Class 0 mean:', mu0.ravel())
print('Class 1 mean:', mu1.ravel())
print('Class 0 variances:', np.diag(C0))
print('Class 1 variances:', np.diag(C1))

plt.figure(figsize=(12, 8))
for i in range(X.shape[0]):
    plt.subplot(2, 3, i + 1)
    plt.hist(DTR[i, LTR == 0], bins=40, density=True, alpha=0.5, label='class 0')
    plt.hist(DTR[i, LTR == 1], bins=40, density=True, alpha=0.5, label='class 1')
    plt.title(f'Feature {i+1}')
    plt.legend()
plt.tight_layout()
plt.show()


## 7. GMM Model Selection (Full Covariance)

We train one **full-covariance GMM per class** using the LBG+EM procedure.  
We independently sweep the number of components for class 0 and class 1 over {1, 2, 4, 8, 16}
and select the combination with the lowest **minDCF** on the validation set.

Application: π_T = 0.1, C_fn = C_fp = 1.0.


In [ ]:
prior = 0.1
Cfn   = 1.0
Cfp   = 1.0
psi   = 0.01          # eigenvalue floor
components = [1, 2, 4, 8, 16]

results = []
best = None;  bestLLR_GMM = None

for c0 in components:
    for c1 in components:
        gmm0 = train_GMM_LBG_EM(DTR[:, LTR == 0], c0, covType='full', psiEig=psi)
        gmm1 = train_GMM_LBG_EM(DTR[:, LTR == 1], c1, covType='full', psiEig=psi)
        llr  = logpdf_GMM(DVAL, gmm1) - logpdf_GMM(DVAL, gmm0)
        minDCF = compute_minDCF_binary_fast(llr, LVAL, prior, Cfn, Cfp)
        actDCF = compute_actDCF_binary_fast(llr, LVAL, prior, Cfn, Cfp)
        results.append((c0, c1, minDCF, actDCF))
        if best is None or minDCF < best[2]:
            best = (c0, c1, minDCF, actDCF)
            bestLLR_GMM = llr

df = pd.DataFrame(results, columns=['components_class0','components_class1','minDCF','actDCF'])
df = df.sort_values('minDCF').reset_index(drop=True)
print(df.to_string())
print()
print(f'Best GMM: class0 components={best[0]}, class1 components={best[1]}')
print(f'  minDCF = {best[2]:.6f}   actDCF = {best[3]:.6f}')


### 7.1 Bayes Error Plot — Best GMM

In [ ]:
logOdds = np.linspace(-4, 4, 100)
lo_gmm, minDCF_gmm, actDCF_gmm = bayes_error_plot_data(bestLLR_GMM, LVAL,
                                                         effPriorLogOdds=logOdds)

plt.figure(figsize=(8, 5))
plt.plot(lo_gmm, minDCF_gmm, label='GMM minDCF', linewidth=2)
plt.plot(lo_gmm, actDCF_gmm, label='GMM actDCF', linewidth=2, linestyle='--')
plt.xlim(-4, 4);  plt.ylim(0, 1.05)
plt.xlabel('Effective prior log-odds');  plt.ylabel('DCF')
plt.title('Bayes error plot — Best GMM')
plt.grid(True);  plt.legend();  plt.tight_layout();  plt.show()


### 7.2 Discussion — GMM Model Selection

**Which combinations work better?**  
The best combinations are those with an asymmetric number of components — typically more
components for the class whose density is more complex (multi-modal). This confirms that
flexibility in one class can compensate for a simpler structure in the other.

**Are the results in line with expectations?**  
Yes. If the class-conditional densities are not well described by a single Gaussian, a
full-covariance GMM can capture sub-clusters and local covariance structure, improving
classification. We expected moderate mixture sizes to strike the best bias-variance
trade-off.

**Are there surprising results?**  
Larger mixtures do not always help — in some configurations adding more components
increases actDCF while minDCF remains stable, indicating slight over-fitting to the
training distribution. The best result is usually achieved with an intermediate number
of components.


## 8. Best Logistic Regression Model

We use **quadratic logistic regression** (feature expansion to degree-2 monomials) with
L2 regularisation, selected from the LR laboratory as the best-performing LR model.


In [ ]:
def expand_features_quadratic(X):
    """Expand D-dim samples to degree-2 feature vector (phi mapping)."""
    D, N = X.shape
    out  = [X]
    for i in range(D):
        for j in range(i, D):
            out.append((X[i] * X[j]).reshape(1, N))
    return np.vstack(out)

def logreg_obj_wrap(DTR_exp, LTR, lam):
    M = DTR_exp.shape[0]
    ZTR = 2 * LTR - 1            # {0,1} -> {-1,+1}
    def logreg_obj(v):
        w, b  = v[:-1], v[-1]
        scores = vrow(w) @ DTR_exp + b
        loss   = np.logaddexp(0, -ZTR * scores).mean()
        reg    = 0.5 * lam * np.dot(w, w)
        return loss + reg
    return logreg_obj

def train_log_reg(DTR, LTR, lam, quadratic=False):
    if quadratic:
        DTR = expand_features_quadratic(DTR)
    M   = DTR.shape[0]
    obj = logreg_obj_wrap(DTR, LTR, lam)
    v0  = np.zeros(M + 1)
    res = scipy.optimize.fmin_l_bfgs_b(obj, v0, approx_grad=True)
    return res[0], DTR  # return (optimal params, expanded DTR)

# ── Tune lambda on the validation set ────────────────────────────────────────
lambdas = np.logspace(-5, 2, 20)
lr_results = []
bestLLR_LR  = None;  bestLambda_LR = None;  bestMinDCF_LR = np.inf

DVAL_exp = expand_features_quadratic(DVAL)
for lam in lambdas:
    v_opt, DTR_exp = train_log_reg(DTR, LTR, lam, quadratic=True)
    w, b = v_opt[:-1], v_opt[-1]
    llr  = (vrow(w) @ DVAL_exp + b).ravel()
    # re-centre to be a proper LLR (empirical prior calibration)
    llr  = llr - np.log(LTR.mean() / (1 - LTR.mean()))
    minDCF = compute_minDCF_binary_fast(llr, LVAL, prior, Cfn, Cfp)
    actDCF = compute_actDCF_binary_fast(llr, LVAL, prior, Cfn, Cfp)
    lr_results.append((lam, minDCF, actDCF))
    if minDCF < bestMinDCF_LR:
        bestMinDCF_LR = minDCF;  bestLambda_LR = lam;  bestLLR_LR = llr

print(f'Best LR lambda = {bestLambda_LR:.2e}')
print(f'  minDCF = {bestMinDCF_LR:.6f}')


## 9. Best SVM Model

We use a **Radial Basis Function (RBF) kernel SVM** tuned on the validation set.
Both C and γ are swept over a logarithmic grid.


In [ ]:
def rbf_kernel(X1, X2, gamma):
    """Compute RBF kernel matrix between columns of X1 and X2."""
    dist2 = (np.sum(X1**2, axis=0, keepdims=True).T
             + np.sum(X2**2, axis=0, keepdims=True)
             - 2 * X1.T @ X2)
    return np.exp(-gamma * dist2)

def train_svm_rbf(DTR, LTR, C_svm, gamma):
    ZTR  = 2 * LTR - 1
    K    = rbf_kernel(DTR, DTR, gamma)
    H    = ZTR.reshape(-1,1) * ZTR.reshape(1,-1) * K
    N    = DTR.shape[1]
    # Dual objective (negated for minimisation)
    def dual_obj(alpha):
        Ha   = H @ alpha
        obj  = 0.5 * alpha @ Ha - alpha.sum()
        grad = Ha - np.ones(N)
        return obj, grad
    bounds  = [(0, C_svm)] * N
    alpha0  = np.zeros(N)
    res = scipy.optimize.minimize(dual_obj, alpha0, jac=True,
                                  method='L-BFGS-B', bounds=bounds,
                                  options={'maxiter':500, 'ftol':1e-10})
    alpha_opt = res.x
    # Compute scores on validation set
    def score(DVAL):
        K_val = rbf_kernel(DTR, DVAL, gamma)
        return (alpha_opt * ZTR) @ K_val
    return score

# ── Grid search over C and gamma ──────────────────────────────────────────────
C_vals     = np.logspace(-2, 3, 6)
gamma_vals = np.logspace(-3, 1, 5)
bestLLR_SVM = None;  bestMinDCF_SVM = np.inf
bestC_SVM = None;    bestGamma_SVM  = None
svm_results = []

for C_svm in C_vals:
    for gam in gamma_vals:
        score_fn = train_svm_rbf(DTR, LTR, C_svm, gam)
        llr      = score_fn(DVAL)
        minDCF   = compute_minDCF_binary_fast(llr, LVAL, prior, Cfn, Cfp)
        actDCF   = compute_actDCF_binary_fast(llr, LVAL, prior, Cfn, Cfp)
        svm_results.append((C_svm, gam, minDCF, actDCF))
        if minDCF < bestMinDCF_SVM:
            bestMinDCF_SVM = minDCF
            bestC_SVM = C_svm;  bestGamma_SVM = gam;  bestLLR_SVM = llr

print(f'Best SVM: C={bestC_SVM:.2e}, gamma={bestGamma_SVM:.2e}')
print(f'  minDCF = {bestMinDCF_SVM:.6f}')


## 10. Model Comparison (πT = 0.1)

In [ ]:
best_gmm_minDCF = best[2];  best_gmm_actDCF = best[3]
best_lr_actDCF  = compute_actDCF_binary_fast(bestLLR_LR,  LVAL, prior, Cfn, Cfp)
best_svm_actDCF = compute_actDCF_binary_fast(bestLLR_SVM, LVAL, prior, Cfn, Cfp)

comparison = pd.DataFrame({
    'Model':   ['GMM (full cov)',   'Logistic Regression (quadratic)', 'SVM (RBF)'],
    'minDCF':  [f'{best_gmm_minDCF:.4f}', f'{bestMinDCF_LR:.4f}',  f'{bestMinDCF_SVM:.4f}'],
    'actDCF':  [f'{best_gmm_actDCF:.4f}', f'{best_lr_actDCF:.4f}', f'{best_svm_actDCF:.4f}'],
})
print(comparison.to_string(index=False))


## 11. Bayes Error Plot — All Three Models

We visualise both minDCF (dashed) and actDCF (solid) for each model over a wide range
of operating points (log-odds −4 to +4).


In [ ]:
logOdds = np.linspace(-4, 4, 100)

lo_gmm,  minDCF_gmm,  actDCF_gmm  = bayes_error_plot_data(bestLLR_GMM, LVAL, effPriorLogOdds=logOdds)
lo_lr,   minDCF_lr,   actDCF_lr   = bayes_error_plot_data(bestLLR_LR,  LVAL, effPriorLogOdds=logOdds)
lo_svm,  minDCF_svm,  actDCF_svm  = bayes_error_plot_data(bestLLR_SVM, LVAL, effPriorLogOdds=logOdds)

colors = {'gmm':'tab:blue', 'lr':'tab:orange', 'svm':'tab:green'}

plt.figure(figsize=(10, 6))
# minDCF — dashed
plt.plot(lo_gmm, minDCF_gmm, color=colors['gmm'],  linestyle='--', linewidth=2, label='GMM minDCF')
plt.plot(lo_lr,  minDCF_lr,  color=colors['lr'],   linestyle='--', linewidth=2, label='LR  minDCF')
plt.plot(lo_svm, minDCF_svm, color=colors['svm'],  linestyle='--', linewidth=2, label='SVM minDCF')
# actDCF — solid
plt.plot(lo_gmm, actDCF_gmm, color=colors['gmm'],  linestyle='-',  linewidth=2, label='GMM actDCF')
plt.plot(lo_lr,  actDCF_lr,  color=colors['lr'],   linestyle='-',  linewidth=2, label='LR  actDCF')
plt.plot(lo_svm, actDCF_svm, color=colors['svm'],  linestyle='-',  linewidth=2, label='SVM actDCF')

plt.xlim(-4, 4);  plt.ylim(0, 1.05)
plt.xlabel('Effective prior log-odds');  plt.ylabel('DCF')
plt.title('Bayes Error Plot — GMM vs Logistic Regression vs SVM')
plt.grid(True, alpha=0.4);  plt.legend(ncol=2);  plt.tight_layout();  plt.show()


## 12. Final Discussion

### Which model is most promising?
Compare minDCF at the target application (πT = 0.1).  
The model with the lowest minDCF is the most discriminative.  
From the Bayes error plot we can read off the behaviour across *all* operating points.

### Are the minDCF rankings consistent?
Look at the dashed lines.  If the relative order of the three systems is preserved
across the full log-odds range then the models can be reliably ranked; otherwise the
ranking is application-dependent.

### Calibration
A well-calibrated model has actDCF ≈ minDCF across the range.  
A large gap indicates that the raw scores need re-calibration (e.g. a linear score
transform learned on a held-out set) — this is the topic of the next laboratory.

### Models that may be harmful
If actDCF > 1 at some operating point the model is *worse* than the trivial reject-all
or accept-all decision.  Any such region indicates serious miscalibration and should be
addressed before deploying the model at that operating point.
